# Parcial 1

Ossian Vladimir Ramirez Hernandez
1975722

---

En esta libreta de ipython incluyo todo el procedimiento realizado a la hora de resolver el parcial. Como el parcial termino siendo para llevar, me extendi en mis procedimientos. El enfoque que tome para resolverlo fue intentar evitar a toda costa usar procedimientos menos optimos (for loops principalmente), y aprovechar las herramientas que numpy y pandas proveen (y que vimos en clase).

La estructura de la libreta es la siguiente:

- Exploracion de datos
- Procedimiento pregunta 1
- Procedimiento pregunta 2
- Procedimiento pregunta 3
- Procedimiento pregunta 4
- Procedimiento pregunta 5
- Procedimiento pregunta 6
- Procedimiento pregunta 7
- Procedimiento pregunta 8

En la misma carpeta que contenga este archivo se encontraran los scripts .py y .txt solicitados en las instrucciones del examen.

> **Nota extra**: Acostumbro escribir en ingles y mi teclado esta en ingles. Para evitarme la molestia de tener que cambiar el lenguaje del teclado y por ende escribir mas lento (pues los caracteres especiales asociados a las teclas cambian), no acentuare palabras ni usare la ñ (usare una simple 'n' en su lugar).

> **Nota extra**: La libreta se realizo usando un kernel de Python 3.13.1

## Exploracion de datos

---

Empezamos importando las librerias que puede utilicemos

In [1]:
# Importar librerias a usar

import pandas as pd
import numpy as np
import re
import math
import matplotlib.pyplot as plt    # Por si me da tiempo a hacer alguna grafica cuando la vea relevante

Antes de proceder, hay que entener las bases de datos con las que contamos.

Para cada base de datos contamos con 5 carpetas con diferentes archivos:

1. Catalogos
    - Esta carpeta contiene archivos .csv que funcionan como catalogos de codigos id. Por ejemplo, en una base de datos puede aparecer el nombre de alguna entidad con un codigo numerico; en la carpeta de catalogos se encuentra el archivo .csv que asocia dicho codigo con el nombre de la entidad.
2. Conjunto_de_datos
    - Esta es la carpeta que contiene las bases de datos principales
3. Diccionario_de_datos
    - Esta carpeta contiene archivos .csv que sirven como diccionario de los campos utilizados en la base de datos principal (e.g. "PAIS_EXTRA" = Pais extranjero). Servira para comprender los campos de las tablas de datos principales.
4. Metadatos
    - Esta carpeta contiene un archivo .txt que tiene una descripcion genera de la base principal
5. Modelo_entidad_relacion
    - Contiene una representacion grafica de la estrutura de los datos (como se relacionan las distintas tablas)

### Base - Museos

Segun los metadatos asociados, tenemos que la primera base consite en "La estadística sobre museos ... sobre la infraestructura de las instituciones museísticas y sus características, el aprovechamiento de estos espacios culturales y las características de operación en relación con su administración. Asimismo, sobre las características sociodemográficas y culturales de sus visitantes y características de la visita ..."

Los dos archivos principales que contienen informacion relevante son:

- [conjunto_de_datos_visita_em2024](../INPUTS/conjunto_de_datos_visita_em2024.csv): Base de datos principal.
- [diccionario_datos_visita_em2024](../INPUTS/diccionario_datos_visita_em2024.csv): Diccionaro con explicaciones de los campos de la base de datos principal.

A continuacion se leen ambos y se muestran en pantalla:


In [2]:
df_museos_bd = pd.read_csv("../INPUTS/conjunto_de_datos_visita_em2024.csv")    # Intente on latin-1 y no procesaba bien algunos caracteres, utf-8 (default) esta mas completo.
df_museos_diccionario = pd.read_csv("../INPUTS/diccionario_Datos_visita_em2024.csv")

display(df_museos_bd.head())

,ANIO_ESTAD,ENT_REGIS,MES_ENTREV,DIA_ENTREV,SEXO,EDAD,ENT_RESID,MUN_RESID,PAIS_RESID,NACIONALID,...,OPIN_EXPOS,NIV_APREND,DUR_VIS_H,DUR_VIS_M,REPETIR_VI,RECOMIE_VI,EVAL_GRAL,M_NOVIS_1,M_NOVIS_2,M_NOVIS_3
0,2024,1,7,1,2,49,11,20.0,0,1,...,2,6.0,0,15,1,13,9,8,NaN,NaN
1,2024,1,7,1,2,40,19,39.0,0,1,...,1,7.0,0,30,1,11,9,7,NaN,NaN
2,2024,1,7,1,2,19,1,1.0,0,1,...,1,10.0,0,15,1,12,10,1,5.0,NaN
3,2024,1,7,1,2,37,1,1.0,0,1,...,1,10.0,0,20,1,11,6,1,9.0,NaN
4,2024,1,7,1,1,42,10,5.0,0,1,...,1,9.0,0,10,1,12,10,8,NaN,NaN


Se puede ver que solo hay valores numericos, lo cual vuelve necesario el uso de los catalogos antes mencionados para traducir el significado. Esto se hara a medida que cada pregunta lo requiera.

Solo para prevenir futuros problemas, hacemos una pequena transformacion a los nombres de columnas, eliminando posibles espacios en blanco al principio y al final de estos.

In [3]:
df_museos_bd.columns = df_museos_bd.columns.astype(str).str.strip()

In [4]:
display(df_museos_diccionario.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   #             70 non-null     int64 
 1   nombre_campo  70 non-null     object
 2   longitud      70 non-null     int64 
 3   tipo          70 non-null     object
 4   nemonico      70 non-null     object
 5   catalogo      70 non-null     object
 6   rango_claves  70 non-null     object
dtypes: int64(2), object(5)
memory usage: 4.0+ KB


None

De usar la funcion .info podemos obsversar que el DataFrame `df_museos_diccionario` tiene 70 columnas. Como es importante entender todos los campos, procedemos a imprimir toda la tabla en dos bloques de codigo (pues en uno solo esta aparece colapsada)

In [5]:
display(df_museos_diccionario.head(35))    # Primera mitad del diccionario

,#,nombre_campo,longitud,tipo,nemonico,catalogo,rango_claves
0,1,Año estadístico,4,C,anio_estad,año_estadistico,2024
1,2,Entidad de registro,2,C,ent_regis,centidad,1..32
2,3,Mes de entrevista,2,N,mes_entrev,meses,1..12
3,4,Día de entrevista,1,N,dia_entrev,dia_entrevista,"1,2,9"
4,5,Sexo,1,N,sexo,condicion_biologica,"1,2"
5,6,Edad,2,N,edad,edad,13..99
6,7,Entidad de residencia habitual,2,C,ent_resid,centidad,"01..33,99"
7,8,Municipio de residencia habitual,3,C,mun_resid,mpio2024,"000..570,999"
8,9,País de residencia habitual en el extranjero,3,C,pais_resid,pais,"000,101..535,998,999"
9,10,Nacionalidad del visitante,1,N,nacionalid,nacionalidad,"1,2,9"


In [6]:
display(df_museos_diccionario.tail(35))    # Segunda mitad del diccionario

,#,nombre_campo,longitud,tipo,nemonico,catalogo,rango_claves
35,36,Motivo de la visita. Otro,1,N,mv_otro,motivo,"0,1"
36,37,Medio de transporte utilizado para llegar al r...,1,N,medio_tran,transporte,"1..7,9"
37,38,Tiempo de traslado,1,N,tiempo_tra,tiempo,1..9
38,39,Tipo de entrada al museo,1,N,tipo_entra,entrada,"1,2,9"
39,40,Persona que acompaña al visitante. Nadie,1,N,pav_nadie,compañia,"0,1"
40,41,Persona que acompaña al visitante. Familia,1,N,pav_famili,compañia,"0,1"
41,42,Persona que acompaña al visitante. Pareja/Novi...,1,N,pav_pareja,compañia,"0,1"
42,43,Persona que acompaña al visitante. Amigos/Cono...,1,N,pav_amigo,compañia,"0,1"
43,44,Persona que acompaña al visitante. Compañeros ...,1,N,pav_compa,compañia,"0,1"
44,45,Persona que acompaña al visitante. Grupo escolar,1,N,pav_escola,compañia,"0,1"


### Base - Poblacion

Segun los metadados asociados, la segunda base de datos se describe como "INEGI. Censo de Población y Vivienda 2020. SNIEG. Información de Interés Nacional.
Conjunto de indicadores sociodemográficos a nivel sección y distrito electoral federal, según la cartografía distrital del INE, provenientes del Censo de Población y Vivienda 2020"

De nuevo, los dos archivos principales que contienen informacion relevante son:

- [INE_SECCION_2020](../INPUTS/INE_SECCION_2020.csv): Base de datos principal
- [Descriptor_indicadores_ECEG_Seccion_2020](../INPUTS/Descriptor_indicadores_ECEG_Seccion_2020.csv): Diccionaro con explicaciones de los campos de la base de datos principal.

A continuacion se leen y se muestran en pantalla:

In [7]:
df_pob_bd = pd.read_csv("../INPUTS/INE_SECCION_2020.csv", encoding="latin-1")
df_pob_diccionario = pd.read_csv("../INPUTS/Descriptor_indicadores_ECEG_Seccion_2020.csv", encoding="latin-1")    # Usando utf-8 marcaba error, cambie a latin-1

display(df_pob_bd.head())

,ID,ENTIDAD,DISTRITO,MUNICIPIO,SECCION,TIPO,POBTOT,POBFEM,POBMAS,P_0A2,...,VPH_TELEF,VPH_CEL,VPH_INTER,VPH_STVP,VPH_SPMVPI,VPH_CVJ,VPH_SINRTV,VPH_SINLTC,VPH_SINCIN,VPH_SINTIC
0,118,1,3,1,1,2,2564,1360,1204,36,...,698,740,738,659,615,288,2,1,12,1
1,122,1,3,1,2,2,889,462,427,27,...,173,253,195,164,145,54,4,20,72,2
2,128,1,3,1,3,2,2003,1057,946,56,...,426,621,559,360,335,184,10,4,74,1
3,137,1,3,1,4,2,1636,880,756,40,...,311,507,456,330,329,147,10,11,61,1
4,138,1,3,1,5,2,808,445,363,14,...,188,256,233,168,119,70,4,0,30,0


Hacemos lo mismo de eliminar espacios en blanco al principio y al final en los nombres de las columnas:

In [8]:
df_pob_bd.columns = df_pob_bd.columns.astype(str).str.strip()

De nuevo, parece que solo hay valores numericos (no es una afirmacion, se necesitaria evaluar para estar seguros). Por lo tanto, va a ser importante el uso de los catalogos asociados para traucir el significado de los codigos numericos.

In [9]:
display(df_pob_diccionario.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226 entries, 0 to 225
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Tema                     226 non-null    object 
 1   Indicador                226 non-null    object 
 2   definicion               226 non-null    object 
 3   Mnemónico                226 non-null    object 
 4   unidad_de_medida         220 non-null    object 
 5   Fuente                   226 non-null    object 
 6   temporalidad             220 non-null    float64
 7   unidad_de_observacion    220 non-null    object 
 8   poblacion_de_referencia  220 non-null    object 
dtypes: float64(1), object(8)
memory usage: 16.0+ KB


None

In [10]:
display(df_pob_diccionario)

,Tema,Indicador,definicion,Mnemónico,unidad_de_medida,Fuente,temporalidad,unidad_de_observacion,poblacion_de_referencia
0,Identificación Geográfica,Consecutivo,Consecutivo único por entidad federativa (no e...,ID,NaN,INEGI. Censo de Población y Vivienda 2020,NaN,NaN,NaN
1,Identificación Geográfica,Clave de entidad federativa,Unidad geográfica mayor de la división polític...,ENTIDAD,NaN,INEGI. Censo de Población y Vivienda 2020,NaN,NaN,NaN
2,Identificación Geográfica,Clave de distrito,Unidad geográfica por distrito electoral federal.,DISTRITO,NaN,INEGI. Censo de Población y Vivienda 2020,NaN,NaN,NaN
3,Identificación Geográfica,Clave de municipio o demarcación territorial,Código que identifica al municipio o demarcaci...,MUNICIPIO,NaN,INEGI. Censo de Población y Vivienda 2020,NaN,NaN,NaN
4,Identificación Geográfica,Clave de sección,Fracción territorial de los distritos electora...,SECCION,NaN,INEGI. Censo de Población y Vivienda 2020,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
221,Vivienda,Viviendas particulares habitadas que disponen ...,Viviendas particulares habitadas que tienen co...,VPH_CVJ,Viviendas,INEGI. Censo de Población y Vivienda 2020,2020.0,Viviendas,Viviendas particulares habitadas que disponen ...
222,Vivienda,Viviendas particulares habitadas sin radio ni ...,Viviendas particulares habitadas que no cuenta...,VPH_SINRTV,Viviendas,INEGI. Censo de Población y Vivienda 2020,2020.0,Viviendas,Viviendas particulares habitadas sin radio ni ...
223,Vivienda,Viviendas particulares habitadas sin línea tel...,Viviendas particulares habitadas que no cuenta...,VPH_SINLTC,Viviendas,INEGI. Censo de Población y Vivienda 2020,2020.0,Viviendas,Viviendas particulares habitadas sin línea tel...
224,Vivienda,Viviendas particulares habitadas sin computado...,Viviendas particulares habitadas que no cuenta...,VPH_SINCIN,Viviendas,INEGI. Censo de Población y Vivienda 2020,2020.0,Viviendas,Viviendas particulares habitadas sin computado...


### Comentarios sobre la calidad de los datos

Observando las tablas anteriores se puede observar una considerable presencia de NaN's. Esto sirve de preambulo, pues ahora sabemos que la tabla precisa de ser limpiada para poder realizar analisis. Si bien estos valores nulos son los unicos detalles visibles, tambien habra que limpiar posibles strings vacios, datos faltantes no NaN, etc. La limpieza de datos se hace en cada pregunta. Se haran copias de las tablas principales y se trabajara en ellas.

## 1ra Pregunta: ¿Cuáles son las 3 nacionalidades extranjeras que mas visitan nuestros museos?

---

Antes de proceder a responder la pregunta y desarrollar el codigo asociado, primero necestiamos comprender que informacion de las bases de datos es relvante, y comprender bien lo que el problema solicita.

### Comprension del problema

De la impresion del diccionario descriptivo en la seccion: [Exploracion de datos](#exploracion-de-datos), podemos observar que hay 5 campos relacionados a visitas. Estos campos tienen los indices `[18,19,20,21,22]`. Debido a que el nombre_campo esta colapsado en algunos, hacemos uso del metodo .iloc (similar a .loc pero usando los indices) para imprimir los valores de dicha columna completos. Hacemos esto para comprender que informacion describen cada uno de esos campos, y asi poder descernir si los datos registrados en ellos importan para la pregunta planteada.


In [11]:
p1_df_museos_diccionario_temp = df_museos_diccionario.copy()    # Acostumbramos a usar copias para evitar mutar la base principal.
p1_df_museos_diccionario_temp = p1_df_museos_diccionario_temp.iloc[[18,19,20,21,22]].loc[:,["nombre_campo", "nemonico"]]
display(p1_df_museos_diccionario_temp)
display(p1_df_museos_diccionario_temp["nombre_campo"].tolist())

,nombre_campo,nemonico
18,Frecuencia de visitas. Primera visita,prim_visit
19,Frecuencia de visitas. Visitas en el año,visit_anio
20,Frecuencia de visitas. Años trascurridos de la...,anios_visi
21,Visitas a otros lugares de exhibición,vis_otros
22,Visitas a otros lugares de exhibición. Cuántos...,viso_cuant


['Frecuencia de visitas. Primera visita',
 'Frecuencia de visitas. Visitas en el año',
 'Frecuencia de visitas. Años trascurridos de la Última visita',
 'Visitas a otros lugares de exhibición',
 'Visitas a otros lugares de exhibición. Cuántos lugares ']

Analizando las descripciones de los campos relacionas con las visitas, podemos observar que hay dos campos que podrian sernos utiles:

1. Frecuencia de visitas. Visitas en el ano : Asociada a la columna "VISIT_ANIO"
2. Visitas a otros lugares de exhibicion. Cuantos lugares: Asociado a la columna "VISO_CUANT"

Claramente lo que nos interesan es la 1ra, aunque pareciera la segunda tambien podria contener infromacion relevante (pues parece puede incluir visitas a otros museos).

#### Analizando "VISO_CUANT"

De la metadata mencionada en la seccion [Exploracion de Datos](#exploracion-de-datos), podemos obtener el link a [Diccionario de Datos](https://www.inegi.org.mx/rnm/index.php/catalog/1101/data-dictionary), donde podemos encontrar una la tabla [vista24](https://www.inegi.org.mx/rnm/index.php/catalog/1101/data-dictionary/F3?file_name=visita24) con las preguntas usadas para realizar la encuesta.

De dicha tabla obtenemos que la pregunta asociada a esta entrada es: 

> ¿En este año ha visitado otros lugares de exhibición como son: museos, galerías, jardines botánicos, zoológicos, acuarios o planetarios? CRUCE UNCÓDIGO
> 
> - 1 Sí
> - Cuántos lugares ____
> - 2 No

Como incluye lugares que no son museos, entonces dicha informacion no nos es util.

#### Analizando "VISIT_ANIO"

De la misma manera, este campo incluye la respuesta a la pregunta

> En los últimos doce meses, ¿cuántas veces ha visitado este lugar, sin incluir la visita de hoy?

En un principio pareciera esta podria ser nuestro campo de interes; aunque a continuacion argumentamos por que no se puede realizar un analisis con esta variable:

#### La falta de ID visitante y la incapacidad usar las visitas acumuladas para responder a la pregunta

Cada entrada en la base de datos esta asociada a una persona que fue entrevistada mientras visitaba un museo, por lo tanto, es posible que una misma persona haya sido encuestada multiples veces al visitar el mismo museo en distintos dias del ano. En caso de que esto pasara, para usar la variable "VISIT_ANIO" para responder a la pregunta, necesitariamos algun proceso para eliminar el riesgo de sobreconteo. Debido a que no tenemos una variable ID_VISTANTE, asumimos realizar esto es inviable. Por lo tanto, tampoco podemos usar la variable "VISIT_ANIO" para responder a esta pregunta.

#### Entonces, como vamos a contar las visitas?

Nos centraremos entonces en registros individuales de visitas. Por lo tanto, la variable de interes sera "PAIS_EXTRA" (pues nos interesan solo las visitas de extranjeros). Del diccionario tenemos que este campo describe:

In [12]:
display(df_museos_diccionario.loc[(df_museos_diccionario["nemonico"].astype(str).str.strip() == "pais_extra"), ["nombre_campo", "nemonico", "rango_claves", "catalogo"]])

,nombre_campo,nemonico,rango_claves,catalogo
10,País de nacionalidad extranjera,pais_extra,"000,101..535,998,999",pais


### **Funcion mapeado clave - valor en catalogo.** 

Observando la anterior tabla podemos ver que los valores en la base de datos principal para esta columna, seran claves numericas. En el catalogo asociado, dichas claves numericas mapean a nombres de paises. Esto significa que al finalizar el procesar los datos para obtener las 3 claves con mayores visitas, tendremos que traducir dichas claves a los nombres de paises usando el catalogo asociado como referencia.

Debido a que este proceso se va a repetir para distintas preguntas, a continuacion definimos una funcion de mapeado clave - valor. La idea sera reutilizarla a lo largo de la libreta.

La funcion debe de tener los siguientes argumentos:

- nombre del catalogo
- DataFrame
- nombre de columna de valores a sustituir

Los archivos de catalogo todos tienen 2 columnas, la primera incluyendo la clave y la segunda el valor que codifica.

A continuacion se define la funcion:

In [13]:
def mapeado_llave_valor_catalogo(_df, _df_catalogo, _nombre_columna_a_sust = str):
    
    _df_catalogo_v2 = _df_catalogo.copy()
    _df_catalogo_v2.index = _df_catalogo_v2[_df_catalogo_v2.columns[0]].tolist()
    _serie_catalogo = _df_catalogo_v2[_df_catalogo_v2.columns[1]]

    _df_v2 = _df.copy()

    _lista_data_a_sust = [_serie_catalogo[x] if (x in _df_catalogo_v2[_df_catalogo_v2.columns[0]].tolist()) else np.nan for x in _df_v2[_nombre_columna_a_sust]]

    _df_v2[_nombre_columna_a_sust] = _lista_data_a_sust
    return _df_v2

# Puede devolver NaNs
    

La fucion cambia los valores indices del dataframe del catalogo (una copia de este, para ser exactos) por las claves. Hacer esto permite el definir un nuevo objeto _serie_catalogo, el cual funcionara como diccionario para parsear las claves a los significados de estas. Al ser una serie, cuando se parseen, el valor de entrada (la llave) se compara con los indices; debido a esto los indices se tuvieron que cambiar por las claves.

Posteriormente se usa list_comprehension para formar la lista _lista_data_a_sust, la cual contendra las traducciones de las claves del dataframe, y tendra NaNs si encuentra valores clave que no se encuentren en el catalogo. Por ultimo, se sustituye la serie asociada a la columna de interes, por la nueva lista con los valores parseados.



Para responder a esta pregunta necesitamos usar los datos en el archivo de las visitas al museo, el catalogo de visitas, el catalogo de nacionalidad, y el catalogo de claves pais.

A continuacion importamos los catalogos correspondientes:

### Resolucion del problema

Ahora si, procedemos a realizar las transformaciones necesarias para obtener las 3 nacionalidades extranjeras que mas visitan nuestros museos.

#### Eliminar NaNs

Un dato NaN en la columna "PAIS_EXTRA" no nos dice nada respecto a la nacionalidad del visitante, por lo que su dato no sirve para los propositos de esta pregunta. El primer paso a realizar es crear una copia de la base de datos de visitantes del museo, y eliminar cualquier entrada que tenga valor NaN en dicha columna. A su vez, podemos de una vez reducir el DataFrame a solo la columna mencionada. Hacemos esto a continuacion:

In [14]:
p1_df_museos_bd_v1 = df_museos_bd.copy()
p1_df_museos_bd_v1 = pd.DataFrame(p1_df_museos_bd_v1.loc[p1_df_museos_bd_v1["PAIS_EXTRA"].notna(), "PAIS_EXTRA"])
display(p1_df_museos_bd_v1)

,PAIS_EXTRA
10,221.0
22,221.0
26,220.0
149,221.0
154,221.0
...,...
180898,0.0
180899,0.0
180900,0.0
180901,0.0


#### Contar visitas por pais extranjero

Ahora contamos las visitas unicas acumuladas por pais extranjero:

In [15]:
p1_df_museos_bd_v2 = p1_df_museos_bd_v1.copy() 
p1_df_museos_bd_v2 = pd.DataFrame(p1_df_museos_bd_v1.groupby("PAIS_EXTRA", as_index=False)["PAIS_EXTRA"].agg(conteo = "count"))
display(p1_df_museos_bd_v2)


,PAIS_EXTRA,conteo
0,0.0,107754
1,101.0,5
2,102.0,2
3,107.0,1
4,117.0,2
...,...,...
91,447.0,5
92,501.0,48
93,520.0,16
94,998.0,26


#### Aplicacion de la funcion mapeado_llave_valor_catalogo

Necesitamos sustituir las claves en la columna "PAIS_EXTRA" con los nombres de los paises que representan. Para ello aplicamos la funcion antes creada, no sin antes leer el archivo que contiene el catalogo asociado (pais, acorde al diccionario ya analizado).

In [16]:
df_catalogo_museos_pais = pd.read_csv("../INPUTS/Pais.csv")
p1_df_museos_bd_v3 = mapeado_llave_valor_catalogo(p1_df_museos_bd_v2, df_catalogo_museos_pais, "PAIS_EXTRA")
display(p1_df_museos_bd_v3)

,PAIS_EXTRA,conteo
0,NaN,107754
1,República de Angola,5
2,República Democrática y Popular de Argelia,2
3,República de Burundi,1
4,República Árabe deEgipto,2
...,...,...
91,Ucrania,5
92,Commonwealth de Australia,48
93,Nueva Zelanda,16
94,Otros países,26


### Resultados finales

Ahora simplemente re-ordenamos el DataFrame, obteniendo las nacionalidades extranjeras que mas visitan nuestros museos

In [17]:
display(p1_df_museos_bd_v3.sort_values(by=["conteo"], ascending=False).head(6))

,PAIS_EXTRA,conteo
0,NaN,107754
27,Estados Unidos de América,3588
95,No especificado,1263
20,República de Colombia,770
70,Reino de España,635
73,República Francesa,454


Si recordamos, la fuincion que utilizamos para traducir las claves, introduce NaN's cuando las claves no se encuentren en el catalogo; de esta manera simplemente ignoramos la entrada NaN. Tambien notese que hay una clave asociada al registro "No especificado"; tampoco lo contamos.

Por lo tanto, las 3 nacionalidades extranjeras que mas visitan nuestros museos (en el periodo 2024) son:

1. Estados Unidos de America
2. Colombia
3. Espana

## 2da Pregunta: ¿Cuáles son las 3 principales ocupaciones del país extranjero que mas visita nuestros museos?

---

### Compresion del problema

De la anterior pregunta sabemos que la nacionalidad extranjera que mas visita nuestros museos, y por ende el pais extranjero que mas los visita, es Estado Unidos. Lo que buscamos responder con esta pregunta es entonces responder a la pregunta: 

> Cuales son las 3 principales ocupaciones en Estado Unidos?

Del analisis exploratorio realizado en la seccion [Exploracion de datos](#exploracion-de-datos), es facilmente identificable que el campo que contendra la informacion relevante sera el de "OCUPACION". Para ver los tipos de valores que contiene y el catalogo asociado a dicha columna, filtramos sobre el diccionario:

In [18]:
display(df_museos_diccionario.loc[df_museos_diccionario["nemonico"] == "ocupacion"], df_museos_diccionario.columns)

,#,nombre_campo,longitud,tipo,nemonico,catalogo,rango_claves
13,14,Ocupación,2,N,ocupacion,ocupacion,"1..11,98 y 99"


Index(['#', 'nombre_campo', 'longitud', 'tipo', 'nemonico', 'catalogo',
       'rango_claves'],
      dtype='object')

Podemos ver de la impresion superior que el catalogo asociado es 'ocupacion' y los valores son claves numericas que habra que traducir usandolo.

### Resolucion del problema

Ahora procedemos con las transformaciones necesarias para obtener las 3 principales ocupaciones de los americanos (de acuerdo a las encuestas realizadas)

#### Filtrar registros: Solo estadounidenses

Solo necesitamos trabajar con los registros de visitantes estadounidenses. Para filtrar ocupamos saber la clave asociada al el pais EUA en la columna "PAIS_EXTRA". Obtenemos dicha clave a continuacion:

In [19]:
display(df_catalogo_museos_pais.loc[df_catalogo_museos_pais[df_catalogo_museos_pais.columns[1]] == "Estados Unidos de América", df_catalogo_museos_pais.columns])

,cve_pais,nombre_pai
77,221,Estados Unidos de América


Ahora que sabemos que la clave pais de EUA es 221, podemos crear una copia de la base de datos de las visitas del museo original, y filtrar usando .loc de nuevo:

In [20]:
p2_df_museos_bd_v1 = df_museos_bd.copy()
p2_df_museos_bd_v1 = p2_df_museos_bd_v1.loc[p2_df_museos_bd_v1["PAIS_EXTRA"] == 221.0, ["PAIS_EXTRA", "OCUPACION"]]
display(p2_df_museos_bd_v1)


,PAIS_EXTRA,OCUPACION
10,221.0,5
22,221.0,2
149,221.0,5
154,221.0,2
193,221.0,4
...,...,...
180221,221.0,4
180286,221.0,11
180305,221.0,5
180308,221.0,3


#### Verificar y remover NaN's 

Antes de remover NaN's, primero revisamos si tan siquiera hay valores de este tipo en el nuevo DataFrame. Usamoa la funcion .info():

In [21]:
display(p2_df_museos_bd_v1.info())

<class 'pandas.core.frame.DataFrame'>
Index: 3588 entries, 10 to 180872
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   PAIS_EXTRA  3588 non-null   float64
 1   OCUPACION   3588 non-null   int64  
dtypes: float64(1), int64(1)
memory usage: 84.1 KB


None

Como hay 3588 valores no nulos, y el dataframe tiene exactamente 3588 entradas, no hay NaNs.

#### Contar cantidad de visitantes por ocupacion

Ahora necesitamos contar la cantidad de visitantos por ocupacion. Realmente la respuesta sera sesgaza. Para una mejor respuesta a la pregunta planteada (las tres principales ocupaciones de los estadounidenses) lo correcto seria analizar una base de datos con informacion laboral de ese Pais. El resultado representa mas una respuesta a la pregunta "Cuales son las 3 principales ocupaciones de los visitantes del pais extranjero que mas visita nuestros museos?"

El procedimiento para obtener los resultados es similar a la pregunta anterior. Hacemos uso de .groupby:

In [22]:
p2_df_museos_bd_v2 = p2_df_museos_bd_v1.copy()
p2_df_museos_bd_v2 =pd.DataFrame(p2_df_museos_bd_v2.groupby("OCUPACION", as_index=False)["OCUPACION"].agg(conteo = "count"))
display(p2_df_museos_bd_v2)

,OCUPACION,conteo
0,1,203
1,2,1437
2,3,86
3,4,210
4,5,110
5,6,6
6,7,93
7,8,57
8,9,38
9,11,983


#### Aplicacion de la funcion mapeado_llave_valor

Ahora tenemos que usar el catalogo 'ocupacion' para traducir las claves y remplazarlas por las ocupaciones que representan. Esto se hace a continuacion:

In [23]:
df_catalogo_museos_ocupacion = pd.read_csv("../INPUTS/ocupacion.csv", encoding="utf-8")    # Por alguna razon ni utf-8 ni latin-1 decodifican bien los caracteres, ut-8 es el que tiene menos errores
p2_df_museos_bd_v3 = mapeado_llave_valor_catalogo(p2_df_museos_bd_v2, df_catalogo_museos_ocupacion, "OCUPACION")
display(p2_df_museos_bd_v3)

,OCUPACION,conteo
0,"Funcionarios, directores y jefes",203
1,Profesionistas y tÃ©cnicos,1437
2,Trabajadores auxiliares en actividades adminis...,86
3,"Comerciantes, empleados en ventas y agentes de...",210
4,Trabajadores en servicios personales y vigilancia,110
5,"Trabajadores en actividades agrÃ­colas, ganade...",6
6,"Trabajadores artesanales, en la construcción y...",93
7,"Operadores de maquinaria industrial, ensamblad...",57
8,Trabajadores en actividades elementales y de a...,38
9,No trabaja,983


### Resultado finales

Odenando el dataframe obtenido de manera descendente obtenemos:

In [24]:
display(p2_df_museos_bd_v3.sort_values(by=["conteo"], ascending=False))

,OCUPACION,conteo
1,Profesionistas y tÃ©cnicos,1437
9,No trabaja,983
10,Insuficientemente especificada,229
3,"Comerciantes, empleados en ventas y agentes de...",210
0,"Funcionarios, directores y jefes",203
11,No especificada,136
4,Trabajadores en servicios personales y vigilancia,110
6,"Trabajadores artesanales, en la construcción y...",93
2,Trabajadores auxiliares en actividades adminis...,86
7,"Operadores de maquinaria industrial, ensamblad...",57


In [25]:
display(p2_df_museos_bd_v3.sort_values(by=["conteo"], ascending=False)["OCUPACION"].head(6).tolist())

['Profesionistas y tÃ©cnicos',
 'No trabaja',
 'Insuficientemente especificada',
 'Comerciantes, empleados en ventas y agentes de ventas',
 'Funcionarios, directores y jefes',
 'No especificada']

Por lo tanto, las tres principales ocupaciones del pais extranjero que mas visita nuestros museos (EUA) son:

1. Profesionistas y tecnicos
2. Comerciantes, empleados en ventas y agentes de ventas
3. Funcionarios, directores y jefes

(Notese ignoramos a los que no trabajan y aquellos que no especificaron lo suficiente como para entrar en una categoria)

## 3ra Pregunta: ¿Cuál estado tiene mas visitantes mexicanos pero de estados externos al del museo?

---

### Comprension del problema

Para poder dar respuesta a la pregunta, tendremos que prestar atencion a los siguientes campos

- "ENT_REGIS": Entidad federativa donde se hizo el registro (y por ende, donde se encuentra el museo)
- "ENT_RESID": Entidad de residencia en la republica mexicana
- "NACIONALID": Nacionalidad Mexicana o extranjera

El campo de nacionalidad lo usaremos para filtrar en caso de que extranjeros que tienen residencia en alguna entidad de Mexico hayan respondido a la encuesta. Dado que solo estamos interesados en Mexicanos, tendremos que eliminar dichos registros. Por otra parte, usaremos las dos columnas restantes para filtrar y contar los visitantes de estados externos al museo.

Para comprender bien estas variables, usamos el diccionario para obtener sus descripciones y catalogos asociados:

In [26]:
display(df_museos_diccionario.loc[df_museos_diccionario["nemonico"].isin(["ent_regis", "ent_resid", "nacionalid"]), df_museos_diccionario.columns])   # Al parecer no fuciona con solo in, con .isin si fuciono

,#,nombre_campo,longitud,tipo,nemonico,catalogo,rango_claves
1,2,Entidad de registro,2,C,ent_regis,centidad,1..32
6,7,Entidad de residencia habitual,2,C,ent_resid,centidad,"01..33,99"
9,10,Nacionalidad del visitante,1,N,nacionalid,nacionalidad,"1,2,9"


De nuevo, todos los valores son claves, sera necesario hacer uso de los catalogos "centidad" y "nacionalidad".

### Resolucion del problema

#### Filtrar por nacionalidad Mexicana

La nacionalidad del visitante esta registrada en el campo "NACIONALID", y este tiene 3 posibles valores: 1,2 o 9. Para entender que significan estas claves, tenemos que leer el catalogo "nacionalidad". Hacemos esto a continuacion:


In [27]:
df_catalogo_museos_nacionalidad = pd.read_csv("../INPUTS/nacionalidad.csv")
display(df_catalogo_museos_nacionalidad)

,clave,descripcion
0,1,Mexicana
1,2,Extranjera
2,9,No especificada


En base al catalogo que se imprimio en el bloque de arriba, el 1 es el que esta asociado a la nacionalidad Mexicana. Ahora que sabemos esto podemos proceder a filtrar la base de datos para solo tener registros de visitas de Mexicanos. Hacemos esto a continuacion:

In [28]:
p3_df_museos_bd_v1 = df_museos_bd.copy()
p3_df_museos_bd_v1 = p3_df_museos_bd_v1.loc[p3_df_museos_bd_v1["NACIONALID"] == 1, ["NACIONALID", "ENT_REGIS", "ENT_RESID"]]
display(p3_df_museos_bd_v1)

,NACIONALID,ENT_REGIS,ENT_RESID
0,1,1,11
1,1,1,19
2,1,1,1
3,1,1,1
4,1,1,10
...,...,...,...
180898,1,32,1
180899,1,32,32
180900,1,32,32
180901,1,32,32


In [29]:
display(p3_df_museos_bd_v1.info())

<class 'pandas.core.frame.DataFrame'>
Index: 170467 entries, 0 to 180902
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   NACIONALID  170467 non-null  int64
 1   ENT_REGIS   170467 non-null  int64
 2   ENT_RESID   170467 non-null  int64
dtypes: int64(3)
memory usage: 5.2 MB


None

Podemos ver del resumen de informacion que no hay datos nulos. No hay que hacer algo al respecto.

#### Ajustar a solo visitas de estados externos al museo

Ya con el dataframe solo conteniendo visitas de mexicanos, ahora usamos la misma funcion .loc para filtrar y eliminar del dataframe aquellas visitas que hicieron mexicanos a museos dentro de su mismo estado de residencia. Como el catalogo que se usa para la entidad de registro y de residencia es la misma, simplemente eliminamos del dataframe aquellos registros con valores repetidos las columnas asociadas a esos datos:

In [30]:
p3_df_museos_bd_v2 = p3_df_museos_bd_v1.copy()
p3_df_museos_bd_v2 = p3_df_museos_bd_v2.loc[p3_df_museos_bd_v2["ENT_REGIS"] != p3_df_museos_bd_v2["ENT_RESID"], ["ENT_REGIS", "ENT_RESID"]]
display(p3_df_museos_bd_v2)

,ENT_REGIS,ENT_RESID
0,1,11
1,1,19
4,1,10
8,1,31
9,1,25
...,...,...
180889,32,8
180890,32,8
180897,32,1
180898,32,1


#### Contar cantidad de visitantes de estados exteriores por estado

Ahora debemos calcular la cantidad de visitas unicas de mexicanos de estados exteriores por estado. Esto se hace utilizando la funcion .groupby():

In [31]:
p3_df_museos_bd_v3 = p3_df_museos_bd_v2.copy()
p3_df_museos_bd_v3 = p3_df_museos_bd_v3.groupby("ENT_REGIS", as_index=False)["ENT_REGIS"].agg(conteo = "count")
display(p3_df_museos_bd_v3)

,ENT_REGIS,conteo
0,1,931
1,2,338
2,3,541
3,4,1160
4,5,1730
5,6,207
6,7,1321
7,8,1652
8,9,13114
9,10,1938


#### Aplicacion de funcion de mapeado

Mismo procedimiento que preguntas anteriores

In [32]:
df_catalogo_museos_centidad = pd.read_csv("../INPUTS/Centidad.csv")
p3_df_museos_bd_v4 = mapeado_llave_valor_catalogo(p3_df_museos_bd_v3, df_catalogo_museos_centidad, "ENT_REGIS")
display(p3_df_museos_bd_v4)

,ENT_REGIS,conteo
0,Aguascalientes,931
1,Baja California,338
2,Baja California Sur,541
3,Campeche,1160
4,Coahuila de Zaragoza,1730
5,Colima,207
6,Chiapas,1321
7,Chihuahua,1652
8,Ciudad de México,13114
9,Durango,1938


### Resultados finales

Para finalizar simplemente tenemos que re-ordenar los registros del ultimo dataframe de mayor a menor

In [33]:
display(p3_df_museos_bd_v4.sort_values(by=["conteo"], ascending=False))

,ENT_REGIS,conteo
8,Ciudad de México,13114
20,Puebla,5307
10,Guanajuato,3975
21,Querétaro,2720
12,Hidalgo,2600
14,México,2541
13,Jalisco,2378
16,Morelos,1966
9,Durango,1938
4,Coahuila de Zaragoza,1730


Por lo tanto, el estado que tiene mas visitantes mexicanos pero de estados externos al del museo es Ciudad de Mexico, con 13,114 de estas visitas registradas durante el periodo 2024 a traves de la encuesta.

## 4ta Pregunta: ¿Cuál es la escolaridad más común de los visitantes mexicanos?

---

### Comprension del problema

Seguimos solo necesitando hacer uso de la tabla de vistas de museos. Los campos que contienen informacion relevante para la pregunta son:

- "ESCOLARIDA": Nivel de escolaridad
- "NACIONALID": Nacionalidad del visitante

Usando el diccionario de la base de visitas a museos, tenemos que ambos de estos campos describen:

In [34]:
display(df_museos_diccionario.loc[df_museos_diccionario["nemonico"].isin(["escolarida", "nacionalid"]), df_museos_diccionario.columns])

,#,nombre_campo,longitud,tipo,nemonico,catalogo,rango_claves
9,10,Nacionalidad del visitante,1,N,nacionalid,nacionalidad,"1,2,9"
11,12,Escolaridad,2,N,escolarida,escolaridad,"1..10,99"


Notese que podremos reutilizar parte del codigo desarrollado para la pregunta anterior, pues en la anterior ya habiamos definido una base que filtraba la nacionalidad, para asi solo tener registros de visitantes mexicanos. 

### Resolucion del problema

#### Filtrar y ajustar las columnas del dataframe

El dataframe previamente definido `p3_df_museos_bd_v1` solo incluye registros de mexicanos, pero no inlcuye la columna "ESCOLARIDA"; usaremos un index.join para juntar dicha columna. Ademas, eliminamos las columnas con valores que no nos son relevantes.

In [35]:
p4_df_museos_bd_v1 = p3_df_museos_bd_v1.copy()
p4_df_museos_bd_v1 = pd.merge(p4_df_museos_bd_v1, pd.DataFrame(df_museos_bd["ESCOLARIDA"]), left_index=True, right_index=True)
p4_df_museos_bd_v1 = pd.DataFrame(p4_df_museos_bd_v1.loc[:, ["ESCOLARIDA"]])
display(p4_df_museos_bd_v1)

,ESCOLARIDA
0,10
1,10
2,8
3,4
4,7
...,...
180898,8
180899,7
180900,4
180901,9


In [36]:
display(p4_df_museos_bd_v1.info())

<class 'pandas.core.frame.DataFrame'>
Index: 170467 entries, 0 to 180902
Data columns (total 1 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   ESCOLARIDA  170467 non-null  int64
dtypes: int64(1)
memory usage: 2.6 MB


None

#### Contar cantidad visitantes mexicanos por escolaridad

Ahora usamos la funcion .groupby para contar la cantidad de visitantes mexicanos por escolaridad

In [37]:
p4_df_museos_bd_v2 = p4_df_museos_bd_v1.copy()
p4_df_museos_bd_v2 = pd.DataFrame(p4_df_museos_bd_v2.groupby("ESCOLARIDA",as_index=False)["ESCOLARIDA"].agg(conteo = "count"))
display(p4_df_museos_bd_v2)

,ESCOLARIDA,conteo
0,1,440
1,2,95
2,3,4976
3,4,17020
4,5,4014
5,6,3514
6,7,38412
7,8,11337
8,9,77699
9,10,12739


#### Aplicacion de funcion de mapeado

Ahora leemos el csv del catalogo "escolaridad", y usamos la funcion antes creada para cambiar las claves en la columna "ESCOLARIDA" por sus significados explicitos.

In [38]:
df_catalogo_museos_escolaridad = pd.read_csv("../INPUTS/escolaridad.csv")
p4_df_museos_bd_v3 = mapeado_llave_valor_catalogo(p4_df_museos_bd_v2, df_catalogo_museos_escolaridad, "ESCOLARIDA")
display(p4_df_museos_bd_v3)

,ESCOLARIDA,conteo
0,Ninguna,440
1,Preescolar,95
2,Primaria,4976
3,Secundaria,17020
4,Estudios técnicos con secundaria terminada,4014
5,Normal básica,3514
6,Preparatoria o bachillerato,38412
7,Estudios técnicos con preparatoria terminada,11337
8,Licenciatura,77699
9,Maestría o doctorado,12739


### Resultados finales

Si bien ya se puede observar cual es la escolaridad mas comun en los visitantes mexicanos, para hacerlo mas obvio simplemente re-ordenamos de mayor a menor y traemos el primer registro.

In [39]:
display(p4_df_museos_bd_v3.sort_values(by=["conteo"], ascending=False).head(1))

,ESCOLARIDA,conteo
8,Licenciatura,77699


Por lo tanto la escolaridad mas comun que tienen los mexicanos que vistan nuestros museos es Licenciatura.

## 5ta Pregunta: ¿Cuál es el estado con mas visitantes de nacionalidad extranjera en proporción a su cantidad de pobladores?

---

### Compresion del prolema

Para poder dar con el estado con mas visitantes de nacionalidad extranjera en proporcion a su cantidad de poladores tendremos que hacer la dos bases de datos principales (la segunda siendo INE_SECCION_2020). Especificamente, tendremos que calcular la cantidad de visitas individuales (recordemos que no es viable analizar visitas acumuladas usando el campo "VISIT_ANIO") de extranjeros por estado, despues agregar hacer un nuevo campo con la razon cantidad_visitas/poblacion.

Notese que podremos reutilizar parte del codigo realizado para la pregunta 1, pues el campo de interes para calcular las vistas de extranjeros es "PAIS_EXTRA" (campo con el cual trabajamos en esa primera pregunta). De la base de datos INE_SECCION_2020 al principio usaremos las columnas ["ENTIDAD", "DISTRITO", "MUNICIPIO", "SECCION", "TIPO", "POBTOT"], aunque parece que solo seran necesarias las columnas "ENTIDAD" y "POBTOT"
 
Usando el diccionario de la base INE_SECCION, tenemos que las descripciones de las columnas mencionadas de esta columna son:

In [40]:
display(df_pob_diccionario.loc[df_pob_diccionario["Mnemónico"].isin(["ENTIDAD", "DISTRITO", "MUNICIPIO", "SECCION", "POBTOT"]), df_pob_diccionario.columns])

,Tema,Indicador,definicion,Mnemónico,unidad_de_medida,Fuente,temporalidad,unidad_de_observacion,poblacion_de_referencia
1,Identificación Geográfica,Clave de entidad federativa,Unidad geográfica mayor de la división polític...,ENTIDAD,NaN,INEGI. Censo de Población y Vivienda 2020,NaN,NaN,NaN
2,Identificación Geográfica,Clave de distrito,Unidad geográfica por distrito electoral federal.,DISTRITO,NaN,INEGI. Censo de Población y Vivienda 2020,NaN,NaN,NaN
3,Identificación Geográfica,Clave de municipio o demarcación territorial,Código que identifica al municipio o demarcaci...,MUNICIPIO,NaN,INEGI. Censo de Población y Vivienda 2020,NaN,NaN,NaN
4,Identificación Geográfica,Clave de sección,Fracción territorial de los distritos electora...,SECCION,NaN,INEGI. Censo de Población y Vivienda 2020,NaN,NaN,NaN
6,Población,Población total,"Personas, nacionales o extranjeras que residen...",POBTOT,Personas,INEGI. Censo de Población y Vivienda 2020,2020.0,Personas,Población total


In [41]:
display(df_pob_diccionario.loc[df_pob_diccionario["Mnemónico"].isin(["ENTIDAD", "DISTRITO", "MUNICIPIO", "SECCION", "POBTOT"]), df_pob_diccionario.columns]["definicion"].tolist())

['Unidad geográfica mayor de la división político-administrativa del país.',
 'Unidad geográfica por distrito electoral federal.',
 'Código que identifica al municipio o demarcación territorial al interior de una entidad federativa, conforme al Marco Geoestadístico.',
 'Fracción territorial de los distritos electorales uninominales para la inscripción de los ciudadanos en el Padrón Electoral y en las Listas Nominales de Electores.',
 'Personas, nacionales o extranjeras que residen habitualmente en el ámbito geográfico de referencia correspondiente. ']

Analizando la tabla recien mostrada y del analisis exploratorio de la seccion [Exploracion de datos](#exploracion-de-datos), podemos intuir que cada entrada de una misma clave en "ENTIDAD" no se repite. Por lo tanto, para calcular la poblacion total por entidad simplemente tendremos que agrupar bajo "ENTIDAD".

### Resolucion del problema

#### Calculo de poblacion total por entidad

Para hacer el calculo de la poblacion total por entidad simplemente tenemos que hacer una copia de la base de datos INE_SECCION_2020, solo trayendo la columna "ENTIDAD" y "POBTOT"; y posterioremente agrupar con respecto a los valores de la columna "ENTIDAD", sumando los valores de la poblacion total asociados. Hacemos esto a conitnuacion:

In [42]:
p5_df_pob_bd_v1 = df_pob_bd.copy()
p5_df_pob_bd_v1 = p5_df_pob_bd_v1.loc[:,["ENTIDAD", "POBTOT"]]
display(p5_df_pob_bd_v1)

,ENTIDAD,POBTOT
0,1,2564
1,1,889
2,1,2003
3,1,1636
4,1,808
...,...,...
68801,32,1027
68802,32,1589
68803,32,269
68804,32,2915


In [43]:
p5_df_pob_bd_v2 = p5_df_pob_bd_v1.copy()
p5_df_pob_bd_v2 = p5_df_pob_bd_v2.groupby("ENTIDAD", as_index=False)["POBTOT"].agg(suma = "sum")
p5_df_pob_bd_v2 = p5_df_pob_bd_v2.rename(columns={"ENTIDAD": "ENT_REGIS", "suma": "suma"})    # Cambiar el nombre de la columna ENTIDAD servira para despues poder hacer un join mas facilmente
display(p5_df_pob_bd_v2)

,ENT_REGIS,suma
0,1,1425607
1,2,3769020
2,3,798447
3,4,928363
4,5,3146771
5,6,731391
6,7,5543828
7,8,3741869
8,9,9209944
9,10,1832650


#### Calculo de visitas totales de extranjeros por entidad

De la pregunta 1 el dataframe `p1_df_museos_bd_v1` ya tiene solo los registros de visitantes extranjeros. Solamente necesitamos usar un index join para agregar la columna "ENT_REGIS". Hacemos esto a continuacion:

In [44]:
p5_df_museos_bd_v1 = p1_df_museos_bd_v1.copy()
p5_df_museos_bd_v1 = pd.merge(p5_df_museos_bd_v1, df_museos_bd.loc[:, "ENT_REGIS"], left_index=True, right_index=True)
display(p5_df_museos_bd_v1)

,PAIS_EXTRA,ENT_REGIS
10,221.0,1
22,221.0,1
26,220.0,1
149,221.0,1
154,221.0,1
...,...,...
180898,0.0,32
180899,0.0,32
180900,0.0,32
180901,0.0,32


Si recordamos del ejercicio 1, las claves 0 y 999 no representan informacion significativa, por lo cual es necesario retirar los registros que tengan estos valores. Hacemos esto a continuacion:

In [45]:
p5_df_museos_bd_v1 = p5_df_museos_bd_v1.loc[(p5_df_museos_bd_v1["PAIS_EXTRA"] != 0.0) & (p5_df_museos_bd_v1["PAIS_EXTRA"] != 999.0), p5_df_museos_bd_v1.columns]
display(p5_df_museos_bd_v1)

,PAIS_EXTRA,ENT_REGIS
10,221.0,1
22,221.0,1
26,220.0,1
149,221.0,1
154,221.0,1
...,...,...
180690,204.0,32
180691,204.0,32
180692,204.0,32
180693,204.0,32


In [46]:
display(p5_df_museos_bd_v1.info())

<class 'pandas.core.frame.DataFrame'>
Index: 8979 entries, 10 to 180872
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   PAIS_EXTRA  8979 non-null   float64
 1   ENT_REGIS   8979 non-null   int64  
dtypes: float64(1), int64(1)
memory usage: 210.4 KB


None

No hay datos nulos. No es necesario filtrar para eliminarlos.

Ahora que ya tenemos el dataframe adecuado, usamos la funcion groupby para agrupar los datos con respecto a "ENT_REGIS", contando los registros de visitas unicas de extranjeros:

In [47]:
p5_df_museos_bd_v2 = p5_df_museos_bd_v1.copy()
p5_df_museos_bd_v2 = pd.DataFrame(p5_df_museos_bd_v2.groupby("ENT_REGIS", as_index=False)["ENT_REGIS"].agg(conteo = "count"))
display(p5_df_museos_bd_v2)
print(p5_df_museos_bd_v2["conteo"].sum()) # Verifico que seran los 8979 registros

,ENT_REGIS,conteo
0,1,50
1,2,97
2,3,52
3,4,72
4,5,110
5,6,24
6,7,239
7,8,365
8,9,2471
9,10,109


8979


#### Merge de ambos dataframes

Ahora tenemos que juntar ambos dataframes, para asi tener tanto el conteo de las visitas de extranjeros por entidad, como la poblacion total. Hacemos un simple merge left:

In [48]:
p5_df_merged_v1 = pd.merge(p5_df_museos_bd_v2, p5_df_pob_bd_v2, how = "left", on="ENT_REGIS")    #Para hacer este merge fue que se cambio el nombre de la columna en pasos anteriores ENTIDAD anteriormente
display(p5_df_merged_v1)

,ENT_REGIS,conteo,suma
0,1,50,1425607
1,2,97,3769020
2,3,52,798447
3,4,72,928363
4,5,110,3146771
5,6,24,731391
6,7,239,5543828
7,8,365,3741869
8,9,2471,9209944
9,10,109,1832650


#### Calculo de proporcion visitas/poblacion

Dado que ya tenemos tanto la cantidad de visitas de extranjeros por entidad y la poblacion total por entidad, ahora podemos crear un nuevo campo que contenga las proporciones visitas/poblacion. Hacemos esto a continuacion:

In [49]:
p5_df_merged_v2 = p5_df_merged_v1.copy()
p5_df_merged_v2["proporcion"] = p5_df_merged_v2["conteo"]/p5_df_merged_v2["suma"]
display(p5_df_merged_v2)

,ENT_REGIS,conteo,suma,proporcion
0,1,50,1425607,0.000035
1,2,97,3769020,0.000026
2,3,52,798447,0.000065
3,4,72,928363,0.000078
4,5,110,3146771,0.000035
5,6,24,731391,0.000033
6,7,239,5543828,0.000043
7,8,365,3741869,0.000098
8,9,2471,9209944,0.000268
9,10,109,1832650,0.000059


#### Aplicacion de funcion de mapeado

Ahora aplicamos la funcion de mapeado para sustituir las claves de entidad por sus nombres explicitos


In [50]:
p5_df_merged_v3 = mapeado_llave_valor_catalogo(p5_df_merged_v2, df_catalogo_museos_centidad, "ENT_REGIS")
display(p5_df_merged_v3)

,ENT_REGIS,conteo,suma,proporcion
0,Aguascalientes,50,1425607,0.000035
1,Baja California,97,3769020,0.000026
2,Baja California Sur,52,798447,0.000065
3,Campeche,72,928363,0.000078
4,Coahuila de Zaragoza,110,3146771,0.000035
5,Colima,24,731391,0.000033
6,Chiapas,239,5543828,0.000043
7,Chihuahua,365,3741869,0.000098
8,Ciudad de México,2471,9209944,0.000268
9,Durango,109,1832650,0.000059


### Resultados finales

Para hacer mas visible cual es la mayor proporcion, hacer uso del metodo .sort_values para ordenarlos de mayor a menor:

In [51]:
display(p5_df_merged_v3.sort_values(by="proporcion").head(1))

,ENT_REGIS,conteo,suma,proporcion
27,Tamaulipas,33,3527735,0.000009


Por lo tanto, el estado con mayor propocion de visitantes extranjeros con respecto a su poblacion es Tamaulipas, con una proporcion de 0.000009 o %0.0009. Los datos tienen sentido, pues tamaulipas comparte frontera con EUA y si recordamos, los estadounidenses son los principales extranjeros que visitan nuestros museos.

## 6ta Pregunta: ¿Cuál es el estado con mas visitantes mexicanos en proporción a su cantidad de pobladores?

---

### Comprension del problema

Esta pregunta es similar a la anterior, solo que ahora solo contabilizaremos visitas unicas de mexicanos. Reutilizaremos el dataframe `p3_df_museos_bd_v1` del ejercicio 3, el cual ya esta filtrado, solo teniendo registros de mexicanos (nacionalidad con codigo 1).

De manera breve, lo que haremos sera agrupar el dataframe antes mencionado con respecto a entidad y asi obtener la suma de visitas por entidad. Posteriormente haremos un left join con el dataframe `p5_df_pob_bd_v2` obtenido en la pregunta anterior.

### Resolucion del problema

#### Calculo de vistas unicas totales de mexicanos por entidad

Hacemos uso de la funcion .groupby para obtener el total de visitas de mexicanos a museos por entidad. Reutilizamos el dataframe `p3_df_museos_bd_v1`.

In [52]:
p6_df_museos_bd_v1 = p3_df_museos_bd_v1.copy()
p6_df_museos_bd_v1 = p6_df_museos_bd_v1.groupby("ENT_REGIS", as_index=False)["NACIONALID"].agg(conteo = "count")
display(p6_df_museos_bd_v1)
print(p6_df_museos_bd_v1["conteo"].sum()) # Verifico que sean los 170467 registros (cantidad de registros de la base p3_df_museos_bd_v1)

,ENT_REGIS,conteo
0,1,2147
1,2,1797
2,3,925
3,4,1873
4,5,5861
5,6,683
6,7,3874
7,8,6020
8,9,33610
9,10,6092


170467


#### Merge de dataframes

Ahora hacemos un left join usando la funcion marge. Juntamos el dataframe recien definido, y el dataframe del ejercicio anterior 'p5_df_pob_bd_v2`.

In [53]:
p6_df_merged_v1 = pd.merge(p6_df_museos_bd_v1, p5_df_pob_bd_v2, how = "left", on = "ENT_REGIS")
display(p6_df_merged_v1)

,ENT_REGIS,conteo,suma
0,1,2147,1425607
1,2,1797,3769020
2,3,925,798447
3,4,1873,928363
4,5,5861,3146771
5,6,683,731391
6,7,3874,5543828
7,8,6020,3741869
8,9,33610,9209944
9,10,6092,1832650


#### Calculo de propocion visitas/poblacion

Identico al ejercicio anterior:

In [54]:
p6_df_merged_v2 = p6_df_merged_v1.copy()
p6_df_merged_v2["proporcion"]= p6_df_merged_v2["conteo"]/p6_df_merged_v2["suma"]
display(p6_df_merged_v2)

,ENT_REGIS,conteo,suma,proporcion
0,1,2147,1425607,0.001506
1,2,1797,3769020,0.000477
2,3,925,798447,0.001158
3,4,1873,928363,0.002018
4,5,5861,3146771,0.001863
5,6,683,731391,0.000934
6,7,3874,5543828,0.000699
7,8,6020,3741869,0.001609
8,9,33610,9209944,0.003649
9,10,6092,1832650,0.003324


#### Aplicacion de funcion de mapeado

Procedimiento repetitivo

In [55]:
p6_df_merged_v3 = mapeado_llave_valor_catalogo(p6_df_merged_v2, df_catalogo_museos_centidad, "ENT_REGIS")
display(p6_df_merged_v3)

,ENT_REGIS,conteo,suma,proporcion
0,Aguascalientes,2147,1425607,0.001506
1,Baja California,1797,3769020,0.000477
2,Baja California Sur,925,798447,0.001158
3,Campeche,1873,928363,0.002018
4,Coahuila de Zaragoza,5861,3146771,0.001863
5,Colima,683,731391,0.000934
6,Chiapas,3874,5543828,0.000699
7,Chihuahua,6020,3741869,0.001609
8,Ciudad de México,33610,9209944,0.003649
9,Durango,6092,1832650,0.003324


### Resultados finales

Finalmente, solo ordenamos de mayor a menor en base a propocion, y mostramos el mayor:

In [56]:
display(p6_df_merged_v3.sort_values(by="proporcion", ascending=False).head(1))

,ENT_REGIS,conteo,suma,proporcion
8,Ciudad de México,33610,9209944,0.003649


Por lo tanto, el estado con mas visitantes mexicanos en proporcion a su poblacion es la ciudad de Mexico, con una proporcion de 0.003649 o %0.3649.

## 7ma Pregunta: ¿Cuál es la nacionalidad extranjera mas común que visita los museos de los 5 estados con mas población?

---

### Comprension del problema

Para obtener la nacionalidad extranjera mas comun que ha visitado los 5 estado con mayor poblacion, lo que haremos sera obtener la cantidad total de visitas unicas por nacionalidad extranjera solo tomando en cuenta registros cuya clave entidad sea la asociada a uno de los 5 estados con mayor poblacion. La nacionalidad que acumule mas visitas, sera nuestra respuesta.

Para esto tendremos que:

1. Obtener los 5 estados con mayor poblacion y sus claves centidad
2. Filtrar la base de datos 'p5_df_museos_bd_v1' (ya filtrada a tener todas las visitas unicas de extranjeros con datos relevantes), solo incluyendo los registros que tengan una de las 5 claves previas
3. Agrupar con respecto a la nacionalidad, sumando las visitas unicas registradas en el dataframe
4. Aplicar funcion mapeo
5. Re-ordenar y obtener el maximo

### Resolucion del problema

#### Obtencion de los 5 estados mas poblados y sus claves

De la [pregunta 5](#5ta-pregunta-cuál-es-el-estado-con-mas-visitantes-de-nacionalidad-extranjera-en-proporción-a-su-cantidad-de-pobladores) tenemos el dataframe `p5_df_pob_bd_v2`, el cual contiene las claves de entidad y las poblaciones totales. Solo reordenamos y obtenemos las claves de los 5 mayores.

In [57]:
p7_df_pob_bd_v1 = p5_df_pob_bd_v2.copy()
p7_df_pob_bd_v1 = p7_df_pob_bd_v1.sort_values(by="suma", ascending=False)
display(p7_df_pob_bd_v1.head(5))

,ENT_REGIS,suma
14,15,16992418
8,9,9209944
13,14,8348151
29,30,8062579
20,21,6583278


Por lo tanto, las claves de los estados/entidades con mayor poblacion son 15, 9, 14, 30 y 21.

#### Filtrado: Solo registros con una de las claves definidas

Ahora simplemente filtramos el dataframe `p5_df_museos_bd_v1` eliminando cualquier registro que no tenga en la columna "ENT_REGIS" una de las 5 claves previamente encontradas.

In [58]:
p7_df_museos_bd_v1 = p5_df_museos_bd_v1.copy()
p7_df_museos_bd_v1 = p7_df_museos_bd_v1.loc[p7_df_museos_bd_v1["ENT_REGIS"].isin([15.0, 9.0, 14.0, 30.0, 21.0]), p7_df_museos_bd_v1.columns]
display(p7_df_museos_bd_v1)
p7_df_museos_bd_v1["ENT_REGIS"].unique() # Verificamos que solo haya valores 9, 14, 15, 21 y 30

,PAIS_EXTRA,ENT_REGIS
24488,402.0,9
24578,204.0,9
24579,204.0,9
24580,215.0,9
24581,215.0,9
...,...,...
174447,213.0,30
174448,213.0,30
174449,213.0,30
174452,225.0,30


array([ 9, 14, 15, 21, 30])

#### Agrupando por pais extranjero

Agrupamos con respecto a los paises extranjeros, sumando los registros de visitas unicas.

In [59]:
p7_df_museos_bd_v2 = p7_df_museos_bd_v1.copy()
p7_df_museos_bd_v2 = p7_df_museos_bd_v2.groupby("PAIS_EXTRA", as_index=False)["ENT_REGIS"].agg(conteo = "count")
display(p7_df_museos_bd_v2)

,PAIS_EXTRA,conteo
0,101.0,3
1,102.0,1
2,107.0,1
3,117.0,2
4,133.0,1
...,...,...
79,446.0,30
80,447.0,1
81,501.0,26
82,520.0,12


#### Aplicacion de funcion mapeado

Trivial.

In [60]:
p7_df_museos_bd_v3 = mapeado_llave_valor_catalogo(p7_df_museos_bd_v2, df_catalogo_museos_pais, "PAIS_EXTRA")
display(p7_df_museos_bd_v3)

,PAIS_EXTRA,conteo
0,República de Angola,3
1,República Democrática y Popular de Argelia,1
2,República de Burundi,1
3,República Árabe deEgipto,2
4,República de Malawi,1
...,...,...
79,Confederación Suiza,30
80,Ucrania,1
81,Commonwealth de Australia,26
82,Nueva Zelanda,12


### Resultados finales

Ahora simplemente ordenamos en base a la columna conteo, de mayor a menor, y traemos el primer registro:

In [61]:
display(p7_df_museos_bd_v3.sort_values(by="conteo", ascending=False).head(1))

,PAIS_EXTRA,conteo
22,Estados Unidos de América,1430


De nuevo, la nacionalidad extranjera estadounidense es la que mas comunmente visita los 5 estados con mayor poblacion. Tiene sentido, depues de todo es la nacionalidad que mas visita nuestros museos.

## 8va Pregunta: ¿Cuál es la escolaridad mas común que visita los museos de los 5 estados con menor población?

### Comprension del problema

Para este problema no discriminamos registros en base a nacionalidad. Para su resolucion tendremos que discriminar entradas con NaNs en la columna de escolaridad, y entradas asociadas a estados que no sean los 5 con menor poblacion total. Lo que haremos sera lo siguiente:

1. Obtener los 5 estados con menor poblacion y sus claves
2. Filtrar la base de datos de visitas de museos, discriminando registros con NaNs en la columna de Escolaridad; y registros que no sean de uno de los 5 estados antes mencioandos
3. Agruparemos por escolaridad, contando la cantidad de visitas unicas
4. Aplicaremos la funcion de mapeado para obtener el nombre explicito de los valores de la columna de escolaridad
5. Re-ordenaremos y obtendremos nuestra respuesta

### Resolucion del problema

#### Obtencion de los 5 estados menos poblados

Para obtener los 5 estados menos poblados, simplemente reutilizamos el dataframe 'p7_df_pob_bd_v1' pero ahora usando el metodo .tails()

In [62]:
display(p7_df_pob_bd_v1.tail(5))

,ENT_REGIS,suma
28,29,1342977
17,18,1235456
3,4,928363
2,3,798447
5,6,731391


Por lo tanto, las claves de los paises con menor poblacion son: 6, 3, 4, 18 y 29

#### Filtrado: Solo registros con una de las claves de entidad y no NaN en escolaridad

Aqui si tendremos que remitirnos a la base de datos original de visitas de mueseos.

In [63]:
p8_df_museos_bd_v1 = df_museos_bd.copy()
p8_df_museos_bd_v1 = p8_df_museos_bd_v1.loc[(p8_df_museos_bd_v1["ESCOLARIDA"].notna()) & (p8_df_museos_bd_v1["ENT_REGIS"].isin([6.0, 3.0, 4.0, 18.0, 29.0])), ["ESCOLARIDA", "ENT_REGIS"]]

display(p8_df_museos_bd_v1)

,ESCOLARIDA,ENT_REGIS
4171,4,3
4172,7,3
4173,7,3
4174,4,3
4175,7,3
...,...,...
171047,10,29
171048,7,29
171049,4,29
171050,9,29


De esta manera ya filtramos por posible valores NaN en la columna "ESCOLARIDA" y solo registros de los 5 estados con menor poblacion.

#### Agrupando por escolaridad

Ahora solo usamos groupby para agrupar por escolaridad, contando todas las visitas unicas por cada clave.

In [64]:
p8_df_museos_bd_v2 = p8_df_museos_bd_v1.copy()
p8_df_museos_bd_v2 = p8_df_museos_bd_v2.groupby("ESCOLARIDA", as_index=False)["ENT_REGIS"].agg(conteo = "count")
display(p8_df_museos_bd_v2)

,ESCOLARIDA,conteo
0,1,14
1,3,153
2,4,640
3,5,193
4,6,170
5,7,1744
6,8,626
7,9,4320
8,10,794
9,99,14


#### Aplicacion de funcion mapeado

Trivial.

In [65]:
p8_df_museos_bd_v3 = mapeado_llave_valor_catalogo(p8_df_museos_bd_v2, df_catalogo_museos_escolaridad, "ESCOLARIDA")
display(p8_df_museos_bd_v3)

,ESCOLARIDA,conteo
0,Ninguna,14
1,Primaria,153
2,Secundaria,640
3,Estudios técnicos con secundaria terminada,193
4,Normal básica,170
5,Preparatoria o bachillerato,1744
6,Estudios técnicos con preparatoria terminada,626
7,Licenciatura,4320
8,Maestría o doctorado,794
9,No especificada,14


### Resultados finales

Ahora simplemente re-ordenamos de mas grande a mas chico en base a la columna conteo, y traemos el primer registro:

In [66]:
display(p8_df_museos_bd_v3.sort_values(by="conteo", ascending=False).head(1))

,ESCOLARIDA,conteo
7,Licenciatura,4320


Por lo tanto la escolaridad mas comun que visita los museos de los 5 estados con menor poblacion es "Licenciatura".